# Stage 13B.6 — Original Spatial-Rule Verification and Final Recovery Gate

**Project:** Malaysia semi-dynamic landslide susceptibility dashboard.

This notebook reconstructs **the exact Stage 04 background eligibility rules** from the original supplied notebook: `EPSG:6933`, `MIN_POSITIVE_BUFFER_M = 2000`, COOLR-specific exclusion radius `max(2000 m, location_accuracy_m)`, and minimum **1000 m** separation between background candidates. It validates Stage 13B.5 assignments against the **actual saved Stage 04 background pool and eligible landslide points**—not merely distances to each matched case.

**Does not overwrite any existing dataset/model.** Temporal approval remains separate: original annual GLC_FCS30D layers may include post-event observations. The provisional 453-row dataset must not be trained until both gates are cleared. The notebook also fixes missing control labels in a *separate audit copy* only.

## PART A — Input files and source policy

In [5]:
%pip install geopandas shapely pyproj pyogrio

from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV = PROJECT_ROOT / "Landslide_FYP_Development"
OUT = DEV / "05_model_results" / "stage13b6_final_recovery_gate"
OUT.mkdir(parents=True, exist_ok=True)

STAGE04_POS = DEV / "02_cleaned_inventory" / "malaysia_positive_inventory_ml_eligible.geojson"
STAGE04_AUDIT = DEV / "01_preprocessing" / "stage04_revised_outputs" / "positive_inventory_with_ml_eligibility.csv"
STAGE04_POOL = DEV / "02_cleaned_inventory" / "background_candidates_3000.csv"
STAGE13B5 = DEV / "05_model_results" / "stage13b5_final_matched_control_validation"
ASSIGN = STAGE13B5 / "provisional_unique_control_assignments.csv"
DATA = STAGE13B5 / "PROVISIONAL_recovered_temporal_matched_groups_NOT_FOR_TRAINING.csv"

# Spatial rules transcribed from original Stage 04 notebook cells 13 and 18.
CRS_METRIC = "EPSG:6933"
MIN_POSITIVE_BUFFER_M = 2000.0
MIN_BACKGROUND_SEPARATION_M = 1000.0
COORDINATE_TOLERANCE_M = 2.0  # compares saved same-source coordinates, not a relaxed exclusion buffer

paths = {"original Stage04 eligible positives":STAGE04_POS,"Stage04 positive audit":STAGE04_AUDIT,
         "original 3000-candidate pool":STAGE04_POOL,"Stage13B5 assignments":ASSIGN,"provisional dataset":DATA}
for name,p in paths.items():
    print(name, p.exists(), p)

# Some Stage04 versions save eligible positives under alternate filenames.
# This is a *strict* fallback: only named source outputs may be used, never guess a substitute dataset.
if not STAGE04_POS.exists():
    candidates = [
        DEV / "02_cleaned_inventory" / "malaysia_positive_inventory_ml_eligible.csv",
        DEV / "02_cleaned_inventory" / "malaysia_positive_inventory_ml_eligible.geojson",
    ]
    for p in candidates:
        if p.exists():
            STAGE04_POS = p
            print("Using Stage04 eligible positives:", p)
            break
if not STAGE04_POS.exists():
    print("Stage04 eligible-positive export not found; Stage04 positive audit may be used only if it contains eligible coordinates.")
for name,p in {"Stage04 positive audit":STAGE04_AUDIT,"original candidate pool":STAGE04_POOL,
               "Stage13B5 assignments":ASSIGN,"provisional dataset":DATA}.items():
    if not p.exists():raise FileNotFoundError(f"Missing {name}: {p}")

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 2.4 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.7 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.7 MB 3.3 MB/s eta 0:00:02
   ---------- ----------------------------- 1.8/6.7 MB 5.1 MB/s eta 0:00:01
   ----------------- ---------------------- 2.9/6.7 MB 5.0 MB/s eta 0:00:01
   ------------------------- -------------- 4.2/6.7 MB 5.4 MB/s eta 0:00:01
   ------------------------------- -------- 5.2/6.7 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------  6.6/6.7 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 6.7/6.7 MB 5.6 MB/s eta 0:00:00
   ------------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


original Stage04 eligible positives True C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.geojson
Stage04 positive audit True C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\stage04_revised_outputs\positive_inventory_with_ml_eligibility.csv
original 3000-candidate pool True C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\background_candidates_3000.csv
Stage13B5 assignments True C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b5_final_matched_control_validation\provisional_unique_control_assignments.csv
provisional dataset True C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b5_final_matched_control_validation\PROVISIONAL_recovered_temporal_matched_groups_NOT_FOR_TRAINING.csv


## PART B — Load original Stage 04 coordinates and reconstruct buffers

In [6]:
def true_mask(s):
    return s.astype("string").str.strip().str.lower().isin(["1","true","yes"])

def load_points(path):
    if path.suffix.lower() in [".geojson", ".gpkg", ".shp"]:
        g = gpd.read_file(path).to_crs("EPSG:4326")
        if "longitude" not in g:g["longitude"] = g.geometry.x
        if "latitude" not in g:g["latitude"] = g.geometry.y
        return g
    d = pd.read_csv(path,low_memory=False)
    if "longitude" not in d or "latitude" not in d:
        raise ValueError(f"Missing lon/lat in {path}")
    return gpd.GeoDataFrame(d,geometry=gpd.points_from_xy(d.longitude,d.latitude),crs="EPSG:4326")

pos_audit = pd.read_csv(STAGE04_AUDIT, low_memory=False)
if STAGE04_POS.exists():
    positives = load_points(STAGE04_POS)
else:
    required = {"longitude","latitude","ml_eligible","source_dataset","location_accuracy_m"}
    if not required.issubset(pos_audit):raise ValueError("Need eligible positive spatial export or Stage04 audit with precise columns: " + str(required-set(pos_audit)))
    positives = load_points(STAGE04_AUDIT)

if "ml_eligible" in positives:
    positives = positives.loc[true_mask(positives["ml_eligible"])].copy()
if "source_dataset" not in positives or "location_accuracy_m" not in positives:
    # Reattach the original Stage04 fields *only* through stable unique identifiers.
    keys=[k for k in ["inventory_id","record_id","landslide_id","sample_id"] if k in positives and k in pos_audit]
    if len(keys)==0:raise ValueError("Stage04 positive geometry missing uncertainty fields and no stable join ID available")
    key=keys[0]
    if positives[key].duplicated().any() or pos_audit[key].duplicated().any():raise ValueError("Positive IDs not unique")
    cols=[key]+[c for c in ["source_dataset","location_accuracy_m"] if c not in positives]
    positives=positives.merge(pos_audit[cols],on=key,validate="one_to_one")

positives["location_accuracy_m"] = pd.to_numeric(positives["location_accuracy_m"],errors="coerce")
positives["exclusion_m"] = MIN_POSITIVE_BUFFER_M
coolr = positives["source_dataset"].astype("string").eq("NASA_COOLR")
if positives.loc[coolr,"location_accuracy_m"].isna().any():
    raise ValueError("Eligible COOLR records with missing uncertainty: cannot verify original buffers")
positives.loc[coolr,"exclusion_m"] = np.maximum(MIN_POSITIVE_BUFFER_M, positives.loc[coolr,"location_accuracy_m"])
positive_metric=positives.to_crs(CRS_METRIC)
print("Stage04 eligible positives:",len(positive_metric))
print("Exclusion radii (m):",positive_metric.exclusion_m.value_counts().sort_index().to_dict())
if len(positive_metric)!=200:
    print("WARNING: Earlier stage recorded 200 positive points; investigate source version before approval.")

pool=load_points(STAGE04_POOL)
assign=pd.read_csv(ASSIGN,low_memory=False)
provisional=pd.read_csv(DATA,low_memory=False)
for frame in [assign,pool,provisional]:
    if "sample_id" in frame:frame["sample_id"]=frame["sample_id"].astype("string").str.strip()
if "background_id" in pool and "sample_id" not in pool:
    pool["sample_id"]=pool["background_id"].astype("string").str.strip()
if "sample_id" not in pool:raise ValueError("Background pool needs sample_id or background_id")
if pool.sample_id.duplicated().any():raise ValueError("Background pool IDs are duplicated")
if assign.sample_id.duplicated().any():raise ValueError("Replacement assignment IDs duplicated")
print("Original background pool:",len(pool),"new control slots:",len(assign),"provisional rows:",len(provisional))

Stage04 eligible positives: 200
Exclusion radii (m): {2000.0: 134, 5000.0: 66}
Original background pool: 3000 new control slots: 31 provisional rows: 453


## PART C — Verify original background pool membership, coordinates and spatial exclusion

In [7]:
# Compare every selected control with its original Stage04 pool coordinate.
cols=["sample_id","longitude","latitude"]
pool_lookup=pool[cols].rename(columns={"longitude":"original_longitude","latitude":"original_latitude"})
check=assign.merge(pool_lookup,on="sample_id",how="left",validate="many_to_one",indicator=True)
for c in ["longitude","latitude","original_longitude","original_latitude"]:
    check[c]=pd.to_numeric(check[c],errors="coerce")
check["original_pool_member"]=check["_merge"].eq("both")
assign_geo=gpd.GeoDataFrame(check,geometry=gpd.points_from_xy(check.longitude,check.latitude),crs="EPSG:4326").to_crs(CRS_METRIC)
pool_match_geo=gpd.GeoDataFrame(check,geometry=gpd.points_from_xy(check.original_longitude,check.original_latitude),crs="EPSG:4326").to_crs(CRS_METRIC)
check["coordinate_displacement_m"]=assign_geo.geometry.distance(pool_match_geo.geometry)
check["original_coordinate_match"]=check["coordinate_displacement_m"].le(COORDINATE_TOLERANCE_M) & check["original_pool_member"]

# Crucially check *every* eligible historical positive; not only the assigned case.
positive_geoms=list(positive_metric.geometry)
radii=positive_metric["exclusion_m"].to_numpy(dtype=float)
nearest_dist=[];nearest_margin=[];outside=[]
for pt in assign_geo.geometry:
    distances=np.fromiter((pt.distance(g) for g in positive_geoms),dtype=float,count=len(positive_geoms))
    margins=distances-radii
    nearest_dist.append(float(np.min(distances)) if len(distances) else np.nan)
    nearest_margin.append(float(np.min(margins)) if len(margins) else np.nan)
    outside.append(bool(np.all(margins>=-0.001)))
check["nearest_eligible_positive_m"]=nearest_dist
check["minimum_buffer_margin_m"]=nearest_margin
check["outside_all_original_positive_buffers"]=outside
check["stage04_spatial_rule_pass"]=(check["original_pool_member"] & check["original_coordinate_match"] & check["outside_all_original_positive_buffers"])
check.drop(columns=["_merge"]).to_csv(OUT/"stage13b6_original_spatial_rule_audit.csv",index=False)
print(check[["sample_id","original_pool_member","coordinate_displacement_m","minimum_buffer_margin_m","stage04_spatial_rule_pass"]].to_string(index=False))
print("Passed Stage04 spatial rules:",int(check.stage04_spatial_rule_pass.sum()),"/",len(check))

sample_id  original_pool_member  coordinate_displacement_m  minimum_buffer_margin_m  stage04_spatial_rule_pass
 BG_00267                  True               0.000000e+00             13523.677485                       True
 BG_00360                  True               5.820766e-11               115.894543                       True
 BG_00547                  True               0.000000e+00             10141.255919                       True
 BG_00206                  True               0.000000e+00              6615.475332                       True
 BG_00236                  True               0.000000e+00             10049.578478                       True
 BG_00343                  True               0.000000e+00              9626.533181                       True
 BG_00127                  True               0.000000e+00             17077.677717                       True
 BG_02901                  True               0.000000e+00             16417.411975                       True
 

## PART D — Global 1 km candidate separation, unique groups, label correction

In [8]:
# Stage 04 required all 3000 generated candidates to be at least 1000 m apart.
# Verify selected controls are valid original pool members AND verify separation from ALL
# candidate positions (not just other recovered controls).
pool_metric=pool.to_crs(CRS_METRIC)
separation=[]
for selected_id, geom in zip(check.sample_id,assign_geo.geometry):
    others=pool_metric.loc[~pool_metric.sample_id.eq(selected_id),"geometry"]
    separation.append(float(others.distance(geom).min()) if len(others) else np.inf)
check["nearest_other_original_candidate_m"]=separation
check["stage04_min_candidate_separation_pass"]=check.nearest_other_original_candidate_m.ge(MIN_BACKGROUND_SEPARATION_M-0.1)
check["all_original_spatial_rules_pass"]=(check.stage04_spatial_rule_pass & check.stage04_min_candidate_separation_pass)
check.drop(columns=["_merge"]).to_csv(OUT/"stage13b6_original_spatial_rule_audit.csv",index=False)

if "sample_class" not in provisional or "temporal_role" not in provisional:
    raise ValueError("Provisional dataset missing sample_class/temporal_role")
if "matched_positive_id" not in provisional:
    raise ValueError("Provisional dataset missing group key")
original_target=pd.to_numeric(provisional.get("target",pd.Series(np.nan,index=provisional.index)),errors="coerce")
provisional["target_before_fix"]=original_target
provisional["target"] = np.where(provisional["temporal_role"].eq("case"),1,
                                 np.where(provisional["temporal_role"].eq("matched_control"),0,np.nan))
unknown_role=provisional.target.isna().sum()
assert unknown_role==0, f"Unknown temporal_role rows: {unknown_role}"
assert not provisional.sample_id.duplicated().any(),"Duplicate sample IDs"
size=provisional.groupby("matched_positive_id").size()
positive_count=provisional.groupby("matched_positive_id")["target"].sum()
structure_ok=bool(size.eq(3).all() and positive_count.eq(1).all())
# Ensure old target values were not altered if already present and valid.
valid_existing=original_target.notna()
assert (provisional.loc[valid_existing,"target"]==original_target.loc[valid_existing]).all(),"Existing nonmissing target conflict"
# This is still an audit-only file; intentionally NOT authorized for training.
provisional.to_csv(OUT/"AUDIT_ONLY_corrected_provisional_labels_NOT_FOR_TRAINING.csv",index=False)
print("Control labels repaired in audit-only copy:",int((~valid_existing & provisional.target.eq(0)).sum()))
print("Complete 1:2 matched-group structure:",structure_ok)
print("Original spatial separation passes:",int(check.all_original_spatial_rules_pass.sum()),"/",len(check))

Control labels repaired in audit-only copy: 302
Complete 1:2 matched-group structure: True
Original spatial separation passes: 31 / 31


## PART E — Historical land cover timing and final gate (conservative)

In [9]:
# Original GLC_FCS30D annual composite cannot be proven strictly pre-event by a year match.
# New pre-event Dynamic World fallback was extracted from a prior 365-day observation window.
# The current replacement assignment table may have revised source in a separate column.
source_col=("revised_landcover_source_candidate" if "revised_landcover_source_candidate" in check
            else "landcover_source")
if source_col not in check:raise ValueError("No land-cover source column in assignments")
check["effective_landcover_source"]=check[source_col].astype("string")
check["is_strict_pre_event_dw"]=check.effective_landcover_source.eq("Dynamic_World_PRE_EVENT_365D")
check["annual_composite_temporal_review_required"]=~check.is_strict_pre_event_dw
check["replacement_candidate_eligible_for_addition"]=(check.all_original_spatial_rules_pass & check.is_strict_pre_event_dw)
check.to_csv(OUT/"stage13b6_replacement_temporal_policy_audit.csv",index=False)

new_groups=set(assign.matched_positive_id.astype(str))
all_new_spatial = bool(check.all_original_spatial_rules_pass.all())
all_new_pre_event = bool(check.is_strict_pre_event_dw.all())
rows=[]
for group,sub in check.groupby("matched_positive_id"):
    rows.append({"matched_positive_id":group,"replacement_slots":len(sub),
                 "all_original_spatial_pass":bool(sub.all_original_spatial_rules_pass.all()),
                 "all_replacement_temporal_strict_pre_event":bool(sub.is_strict_pre_event_dw.all()),
                 "original_group_recovery_approved":bool(sub.replacement_candidate_eligible_for_addition.all())})
group_gate=pd.DataFrame(rows)
group_gate.to_csv(OUT/"stage13b6_group_gate.csv",index=False)

# Current temporal model includes older annual data as well. Clear the global historical-landcover
# validity policy separately before making causal/dynamic claims; changing it may alter the dataset.
ORIGINAL_121_GROUPS_HISTORICAL_LANDCOVER_POLICY_CLEARED = False
all_gate = (all_new_spatial and all_new_pre_event and structure_ok
            and ORIGINAL_121_GROUPS_HISTORICAL_LANDCOVER_POLICY_CLEARED)
checks=[
 ("Stage04 original pool membership + point buffers", "PASS" if all_new_spatial else "FAIL", "Original EPSG:6933; max(2km, COOLR uncertainty); 1km pool separation"),
 ("Unique control IDs and full 1:2 groups", "PASS" if structure_ok else "FAIL", "All original sample IDs should be unique"),
 ("Correction of background target labels", "PASS", "Audit-only copy; background target=0"),
 ("All selected replacements strictly pre-event landcover", "PASS" if all_new_pre_event else "REVIEW_REQUIRED", "Annual GLC map may be post-event"),
 ("Original 121 group temporal policy cleared", "PASS" if ORIGINAL_121_GROUPS_HISTORICAL_LANDCOVER_POLICY_CLEARED else "PENDING", "Audit prior annual composites and their temporal representation"),
 ("Approved recovered dataset for retraining", "AUTHORIZED" if all_gate else "NOT_AUTHORIZED", "Rerun three-algorithm nested spatial comparison after data approval"),
]
pd.DataFrame(checks,columns=["check","status","detail"]).to_csv(OUT/"stage13b6_decision_gate.csv",index=False)
summary={"replacement_slots":int(len(check)),"provisional_recovered_groups":int(len(new_groups)),
         "stage04_spatial_pass_slots":int(check.all_original_spatial_rules_pass.sum()),
         "strict_pre_event_dw_replacement_slots":int(check.is_strict_pre_event_dw.sum()),
         "groups_passing_replacement_spatial_and_temporal":int(group_gate.original_group_recovery_approved.sum()),
         "provisional_total_cases":int(provisional.target.eq(1).sum()),
         "provisional_total_controls":int(provisional.target.eq(0).sum()),
         "annual_composite_global_policy_cleared":ORIGINAL_121_GROUPS_HISTORICAL_LANDCOVER_POLICY_CLEARED,
         "retraining_authorized":bool(all_gate),
         "note":"Original complete temporal dataset and prior models remain unchanged; no retraining without full temporal policy and spatial dataset validation."}
(OUT/"stage13b6_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))
print("Outputs:",OUT)

{
  "replacement_slots": 31,
  "provisional_recovered_groups": 30,
  "stage04_spatial_pass_slots": 31,
  "strict_pre_event_dw_replacement_slots": 24,
  "groups_passing_replacement_spatial_and_temporal": 23,
  "provisional_total_cases": 151,
  "provisional_total_controls": 302,
  "annual_composite_global_policy_cleared": false,
  "retraining_authorized": false,
  "note": "Original complete temporal dataset and prior models remain unchanged; no retraining without full temporal policy and spatial dataset validation."
}
Outputs: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b6_final_recovery_gate


## What to send for review

Upload: `stage13b6_summary.json`, `stage13b6_decision_gate.csv`, `stage13b6_original_spatial_rule_audit.csv`, `stage13b6_replacement_temporal_policy_audit.csv`, `stage13b6_group_gate.csv`. The audit-only corrected CSV can be omitted initially.

**Interpretation:** Spatial pass confirms rules from the exact original Stage 04 candidate pool. It does **not** certify the historical timing of all annual land-cover composites. A record's year-matched annual map is not by itself proof that all imagery predates the landslide.

**After approval:** Reconstruct the final model-ready table, verify all static predictor columns and group folds, rerun LR/RF/XGBoost nested spatial benchmarking, then train replacement production pipelines. Do not silently reuse previous test results.